# analyze wave signals

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pylab as plt
import datetime as dt
import glob
import os
import xarray as xr
import numpy as np

from mpl_toolkits.basemap import Basemap

In [2]:
# keepdat=["device_id","UTC_datetime","UTC_date","UTC_time","datatype","satcount","Latitude","Longitude","depth_m",
#          "conductivity_mS/cm","ext_temperature_C"]

# keepdat=["device_id","UTC_datetime","UTC_date","UTC_time","datatype","satcount","Latitude","Longitude","depth_m",
#          "conductivity_mS/cm","ext_temperature_C"]

keepdat=["device_id","UTC_datetime","UTC_date","UTC_time","datatype","satcount","Latitude","Longitude","depth_m","acc_x","acc_y","acc_z","milliseconds","UTC_timestamp"
         ]

## load Field_Data file

In [3]:
##### show all the columns of the Pandas DataFrame
# pd.set_option('display.max_columns',None)

def pd_read_pattern(pattern):
    files = glob.glob(pattern)
 
    dfs = pd.DataFrame()
#     print(dfs)
    for f in files:
        
        print (f)
        
        dfs = dfs.append(pd.read_csv(f,parse_dates=True,index_col='UTC_datetime',
                          usecols=keepdat))
        
        # dfs.drop(dfs[dfs['datatype'] == 'SEN_ACC_5Hz'].index, inplace = True)
        # dfs.drop(dfs[dfs['datatype'] == 'SEN_ACC_5Hz_END'].index, inplace = True)
        # dfs.drop(dfs[dfs['datatype'] == 'SEN_ACC_5Hz_START'].index, inplace = True)
        # dfs.drop(dfs[dfs['datatype'] == 'SEN_ACC_5Hz_ENDINT'].index, inplace = True)
        dfs.drop(dfs[dfs['datatype'] == 'SEND_ALL2_1Hz'].index, inplace = True)
        dfs.drop(dfs[dfs['datatype'] == 'SEND_ALL2_1Hz_START'].index, inplace = True)
        dfs.drop(dfs[dfs['datatype'] == 'SEND_ALL2_1Hz_END'].index, inplace = True)
        
# keep the GPS lines to get lon/lat for the location of birds
        # dfs.drop(dfs[dfs['datatype'] == 'GPS'].index, inplace = True)
        # dfs.drop(dfs[dfs['datatype'] == 'GPSS'].index, inplace = True)
        # dfs.drop(dfs[dfs['datatype'] == 'GPSD'].index, inplace = True))
        # dfs.drop(dfs[dfs['datatype'] == 'GPSF'].index, inplace = True)

    return dfs #.reset_index(drop=True)

In [4]:
# get IDs observed in UAEBUS///
# idfs = pd.DataFrame()
idfile = '/home/server/pi/homes/liux8/work/mnt/Field_Data/Deployment_Field_Data.csv'
IDS = pd.read_csv(idfile,encoding='latin-1')
IDS.drop(IDS[IDS['Project_ID'] != 'PERIGGU25'].index, inplace = True)
IDS.columns
# kuae = IDS(strcmp(IDS.Var4,'UAEBUSO20'),:);
# idnumber0 = [];
# for i = 1:size(kuae.Var14,1)
#    idnumber0 = kuae.Var14{i};
#    idnum{i} = idnumber0(:,end-5:end);
# end
# id_total = size(idnum,1);
# IDS['Deployment_ID'].shape[0]

Index(['ProjectTitle', 'PIEmail', 'DataManagerEmail', 'Project_ID', 'Bird_ID',
       'Species_Code', 'Species', 'Capture_Site', 'Capture_Country',
       'Capture_Number', 'Capture_Date', 'Capture_Time', 'Release_Date',
       'Release_Time', 'Deployment_ID', 'DeploymentStartDatetime_Local',
       'UTC_offset_deploy', 'DeploymentLatitude', 'DeploymentLongitude',
       'TagManufacture', 'TagModel', 'Sensors', 'TagSerialNumber', 'USGS_Band',
       'Aux_Band_Code', 'Aux_Band_Color', 'Tarsus_mm', 'Culmen_mm',
       'Wing_Chord_cm', 'Total_Weight_kg', 'Bag_Weight_kg', 'Bird_Weight_kg',
       'Sex_Mass', 'Feathers_YN', 'AnimalAgeClass', 'Plumage_Notes', 'Notes',
       'DeploymentEndDatetime_UTC', 'Deployment_End_Notes',
       'Deployment_End_Short', 'DeploymentEndConfirmedYN',
       'DataEmbargoUntilDate', 'Status', 'Tagging_person', 'Breeding_Status',
       'Nest_Contents', 'Recapture_Y/N', 'Recapture_Date', 'Recapture_Time',
       'ReRelease_Date', 'ReRelease_Time', 'Recapture_B

## load tag data

In [5]:
# ID_list_PERIGGU25= [225660,225661,225663,225665]
# ID_list_PERIGGU25_1 = [225661]
ID_list_PERIMGU25 = [225660, 225661, 225663, 225665]

In [6]:
dfs = pd.DataFrame()
# for iid in range(IDS['Deployment_ID'].shape[0]):
# for iid in range(1):
#     # id = IDS['Deployment_ID'].values[iid][-6:]
#     id = ID_list_PERIGGU25_1[iid]

#     print(id)
#     # dfs = dfs.append(pd_read_pattern('/home/server/pi/homes/liux8/work/mnt/bird_ATN_files/'+IDS['Project_ID'].values[iid]+'/gps_sensors_v2/'+id+'_20??_[0-1][0-9].csv'))
#     dfs = dfs.append(pd_read_pattern('/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/'+str(id)+'_2025_[0-1][0-9].csv'))
#     if not dfs.empty:
#         # dfs=dfs.loc['2021-01-01':'2021-12-31']

#         dfs['date']=pd.to_datetime(dfs['UTC_date']+' '+dfs['UTC_time'], format='%Y-%m-%d %H:%M:%S')

#         threshold = pd.Timedelta(minutes=3)
#     print(dfs['device_id'][0:5])

for iid in range(4):
    # id = IDS['Deployment_ID'].values[iid][-6:]
    id = ID_list_PERIMGU25[iid]

    print(id)
    # dfs = dfs.append(pd_read_pattern('/home/server/pi/homes/liux8/work/mnt/bird_ATN_files/'+IDS['Project_ID'].values[iid]+'/gps_sensors_v2/'+id+'_20??_[0-1][0-9].csv'))
    dfs = dfs.append(pd_read_pattern('/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/'+str(id)+'_20??_[0-1][0-9].csv'))
    if not dfs.empty:
        # dfs=dfs.loc['2021-01-01':'2021-12-31']

        dfs['date']=pd.to_datetime(dfs['UTC_date']+' '+dfs['UTC_time'], format='%Y-%m-%d %H:%M:%S')

        threshold = pd.Timedelta(minutes=3)
    print(dfs['device_id'][0:5])

225660
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2022_10.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_01.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_02.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_03.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_04.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_05.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225660_2025_06.csv


/tmp/ipykernel_2872802/482197327.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = dfs.append(pd.read_csv(f,parse_dates=True,index_col='UTC_datetime',


UTC_datetime
2022-10-05 19:15:48    225660
2022-10-14 21:58:36    225660
2022-10-14 22:12:59    225660
2022-10-14 22:28:01    225660
2022-10-14 22:43:03    225660
Name: device_id, dtype: int64
225661
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2022_10.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_01.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_02.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_03.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_04.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_05.csv
/home/server/pi/homes/liux8/work/mnt/2019_BRAC_Ornitela/raw_csv_monthly/225661_2025_06.csv
UTC_datetime
2022-10-05 19:15:48    225660
2022-10-14 21:58:36    225660
2022-10-14 22:12:59    225660
2022-10-14 22:28:01    225660
2022-10-14 22:43:03    225660


/tmp/ipykernel_2872802/482197327.py:13: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = dfs.append(pd.read_csv(f,parse_dates=True,index_col='UTC_datetime',


UTC_datetime
2022-10-05 19:15:48    225660
2022-10-14 21:58:36    225660
2022-10-14 22:12:59    225660
2022-10-14 22:28:01    225660
2022-10-14 22:43:03    225660
Name: device_id, dtype: int64


In [7]:
# dfs[(dfs['date']>=pd.to_datetime('2025-1-15 3:36'))&(dfs['date']<=pd.to_datetime('2020-5-15 9:42'))&(dfs['device_id']==191991)]
dfs

,device_id,UTC_date,UTC_time,datatype,satcount,Latitude,Longitude,acc_x,acc_y,acc_z,UTC_timestamp,milliseconds,depth_m,date
UTC_datetime,,,,,,,,,,,,,,
2022-10-05 19:15:48,225660,2022-10-05,19:15:48,GPS,9.0,54.784237,25.192787,25.0,73.0,1083.0,2022-10-05 19:15:48.485,485,NaN,2022-10-05 19:15:48
2022-10-14 21:58:36,225660,2022-10-14,21:58:36,GPS,9.0,44.558445,-123.271347,16.0,26.0,1068.0,2022-10-14 21:58:36.000,0,NaN,2022-10-14 21:58:36
2022-10-14 22:12:59,225660,2022-10-14,22:12:59,GPS,8.0,44.558456,-123.271378,16.0,25.0,1071.0,2022-10-14 22:12:59.602,602,NaN,2022-10-14 22:12:59
2022-10-14 22:28:01,225660,2022-10-14,22:28:01,GPS,7.0,44.558617,-123.271515,13.0,23.0,1076.0,2022-10-14 22:28:01.622,622,NaN,2022-10-14 22:28:01
2022-10-14 22:43:03,225660,2022-10-14,22:43:03,GPS,6.0,44.558559,-123.271057,20.0,24.0,1071.0,2022-10-14 22:43:03.596,596,NaN,2022-10-14 22:43:03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-23 19:18:37,225665,2025-06-23,19:18:37,SEN_ACC_5Hz,NaN,NaN,NaN,-5.0,170.0,1023.0,2025-06-23 19:18:37.720,720,NaN,2025-06-23 19:18:37
2025-06-23 19:18:37,225665,2025-06-23,19:18:37,SEN_ACC_5Hz,NaN,NaN,NaN,37.0,150.0,1027.0,2025-06-23 19:18:37.920,920,NaN,2025-06-23 19:18:37
2025-06-23 19:18:38,225665,2025-06-23,19:18:38,SEN_ACC_5Hz_END,NaN,NaN,NaN,49.0,167.0,894.0,2025-06-23 19:18:38.120,120,NaN,2025-06-23 19:18:38


In [8]:
dfs['device_id'].unique()

array([225660, 225661, 225663, 225665])

In [ ]:
dfs.to_csv('~/work/OBS/dashcams_data/peru_data_acc/wave_acc/PERU_acc_raw_longrec_keepGPS_4tags_June27.csv')